# Milestone 1 Lab - The mocked business answer

## Using AI to answer a retail profit question

Kestrel Ridge Outfitters is a fictional online retailer of outdoor gear. A retail leader wants to ask a profit question in plain language and get an answer they can trust.

In this lab we use AI to meet that business requirement, and we measure how well it does. First we hand a model all the data and ask the question directly. Then we give an agent a set of tools and ask again. We score both against answers we already know are correct.

Along the way the lab shows you the three design documents for this milestone, each at the point where you need it:

- **BRD** (business requirements): what the business needs and why
- **HLD** (high-level design): the parts of the system and how data moves between them
- **LLD** (low-level design): the exact tables, formulas, tools, and checks

## 6 parts:

PART 1: The business brief

PART 2: Examine the data

PART 3: Ask AI to draft the metric definitions

PART 4: One model call, with all the data

PART 5: An agent with tools

PART 6: Verify, compare, and move to production

> ### Before you start
>
> This lab uses `openrouter/free` throughout, so it needs no credit on OpenRouter. It reads your key from the `.env` file in the project root (copy `.env.example` to `.env` and add your `OPENROUTER_API_KEY`).
>
> OpenRouter's free models allow about 50 requests a day on an account without credit. With `TINY = True`, the default here, this lab makes roughly 25. `openrouter/free` picks a different free model for each request, so expect your results to vary from run to run.
>
> Without a key, the cells that call a model are skipped and everything else still runs.
>
> Please execute each cell and read the output before moving on. Add print statements and try small changes to make the code your own.

In [1]:
import json
import os
import re
import sys
import time
from pathlib import Path
from typing import Literal

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from pydantic import BaseModel, Field

## Find this milestone's folder

Each milestone is self-contained. Everything this lab reads is inside `src/m1_mocked_business_answer/`: the documents in `docs/`, the data in `data/mock/`, and the evaluation cases in `evals/`.

`show_doc` displays one section of a design document, so you read the same text that lives in the `docs/` folder. The diagrams in those files are written in Mermaid, which notebook output can't draw, so `show_doc` points you to the file for those.

In [2]:
MILESTONE = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "docs" / "BRD.md").exists())
DOCS = MILESTONE / "docs"
DATA = MILESTONE / "data" / "mock"
EVALS = MILESTONE / "evals"


def get_section(document, heading):
    """Return one '## ' section of a design document as text."""
    text = (DOCS / document).read_text(encoding="utf-8")
    match = re.search(rf"^## {re.escape(heading)}\n(.*?)(?=^## |\Z)", text, flags=re.DOTALL | re.MULTILINE)
    if not match:
        raise ValueError(f"No section '{heading}' in {document}")
    return match.group(1).strip()


def show_doc(document, heading):
    """Display one section of a design document."""
    body = re.sub(r"```mermaid.*?```", f"*(Diagram: open `docs/{document}` to see it.)*", get_section(document, heading), flags=re.DOTALL)
    display(Markdown(f"> **From `docs/{document}`: {heading}**\n\n{body}"))


print(f"Milestone folder: {MILESTONE.name}")

Milestone folder: m1_mocked_business_answer


## Part 1: The business brief

Before any code, read what the business is asking for. A real project starts here, and so do we.

This is the **BRD**. It uses business language only: no tables, no tools, no models.

In [3]:
show_doc("BRD.md", "1. Business problem")

> **From `docs/BRD.md`: 1. Business problem**

Kestrel Ridge Outfitters is a fictional online retailer of outdoor gear. Its leaders can see sales, but they cannot easily see which products actually make money once discounts, refunds, product cost, and delivery costs are taken out. When they do get a profit figure, they cannot tell where it came from or whether to trust it.

They want to ask a profit question in plain language and get an answer that shows its working.

Here is the question we have to answer. Notice that it's really three questions: which products and categories, what changed, and how confident should I be.

In [4]:
show_doc("BRD.md", "3. The question to answer")

> **From `docs/BRD.md`: 3. The question to answer**

> Which products and categories generated the most contribution profit last month, what changed, and how confident should I be in those numbers?

The analyst must also handle the smaller questions inside it:

- What was a given profit metric for a product or category in a given month?
- How did it change from the month before?
- How is a metric defined?
- Is there a known data problem behind a figure?

And this is what a good answer contains. Keep this list in mind: in Parts 4 and 5 we'll check the AI's answers against it.

In [5]:
show_doc("BRD.md", "5. What a good answer contains")
show_doc("BRD.md", "9. Success criteria")

> **From `docs/BRD.md`: 5. What a good answer contains**

1. A headline that answers the question in one or two sentences.
2. The figures, each with the month, the product or category, and where it came from.
3. The metric definition used, with its version.
4. The assumptions that shape the figures.
5. Any known data problem that affects those figures.
6. A confidence level (high, medium, or low) with the reasons for it.

> **From `docs/BRD.md`: 9. Success criteria**

- Every figure in an answer can be traced to a stored value. None is produced by the AI model.
- Every answer names its sources and the metric definition version.
- The answer to the main question names the top products and categories, describes the change from September, and gives a confidence level with reasons.
- A known data problem behind a figure is mentioned and lowers the confidence.
- A question outside the scope above is declined with an explanation.

## Connect to OpenRouter

The OpenRouter key comes from the `.env` file in the project root.

### NOTE:

If this is a new OpenRouter account, you might get an error like "new accounts are limited to 20 requests per minute for this model". If so, add `time.sleep(3)` immediately after each call to a model.

In [6]:
load_dotenv(find_dotenv(usecwd=True), override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
MODEL = os.getenv("MODEL_NAME", "openrouter/free")
LIVE = bool(openrouter_api_key)
SKIPPED = "Skipped: no OpenRouter key. Add OPENROUTER_API_KEY to the .env file in the project root to run this cell."

if LIVE:
    print(f"OpenRouter API key found. Using model {MODEL}")
    openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)
else:
    print("No OpenRouter API key found. The cells that call a model will be skipped.")

OpenRouter API key found. Using model openrouter/free


> ### Reminder: the Chat Completions API
>
> We call models with the Chat Completions API. `messages` is a Python list of dicts, each with a `role` and `content`. The system prompt tells the model how to behave and the user prompt has the request. The reply is in `response.choices[0].message.content`.
>
> Here's a quick test.

In [7]:
messages = [
    {"role": "system", "content": "You are a retail analyst who answers in a single short sentence."},
    {"role": "user", "content": "Why is profit a better measure than sales?"},
]

if LIVE:
    response = openrouter.chat.completions.create(model=MODEL, messages=messages)
    print(f"Answered by {response.model}")
    display(Markdown(response.choices[0].message.content))
else:
    print(SKIPPED)

Answered by nvidia/nemotron-3.5-super-vl:free


Profit shows actual earnings after expenses, while sales only measure total revenue.

## Part 2: Load and examine the data

The mock data is in this milestone's `data/mock` folder. It is hand-prepared: nobody calculated it from real orders. Milestone 2 replaces it with real calculations.

There are four small tables. This is the first time we need the **LLD**, which gives the exact columns of each one.

In [8]:
show_doc("LLD.md", "3. Mock tables")

> **From `docs/LLD.md`: 3. Mock tables**

**`products.csv`** (12 rows)

| Column | Type | Notes |
|---|---|---|
| `product_id` | text | `P001` to `P012`, unique |
| `product_name` | text | |
| `category` | text | `Hydration`, `Camp`, or `Apparel`; 4 products each |

**`monthly_product_profit.csv`** (24 rows; one per product per month)

| Column | Type | Notes |
|---|---|---|
| `month` | text | `2026-09` or `2026-10` |
| `product_id` | text | Must exist in `products.csv` |
| `units_sold` | integer | |
| `gross_sales`, `discounts`, `refunds`, `net_sales` | decimal | |
| `product_cost`, `gross_profit` | decimal, nullable | Null when cost is missing |
| `fulfillment_cost`, `shipping_cost`, `payment_fees`, `return_cost` | decimal | |
| `contribution_profit` | decimal, nullable | Null when `gross_profit` is null |
| `metric_version` | text | `1.0.0` |

Every row must satisfy the formulas in section 2. A test checks this.

**`evidence_records.csv`** (about 3 per product per month)

| Column | Type | Notes |
|---|---|---|
| `evidence_id` | text | `EV-0001`, unique |
| `month`, `product_id` | text | |
| `record_type` | text | `order_item`, `refund`, `product_cost`, `payment_fee`, or `shipment` |
| `source_table` | text | The table the record would come from, for example `orders` |
| `source_record_id` | text | For example `ORD-10231` |
| `amount` | decimal | |
| `description` | text | |

These are samples. They illustrate a figure and do not add up to it. Every tool result that returns them says so.

**`quality_flags.csv`** (3 planted rows)

| Column | Type | Notes |
|---|---|---|
| `flag_id` | text | `QF-001`, unique |
| `month` | text | |
| `product_id` | text | |
| `check_name` | text | |
| `severity` | text | `high`, `medium`, or `low` |
| `affected_metrics` | text | Metric names separated by `;` |
| `description` | text | |

| Flag | Problem | Severity | Effect |
|---|---|---|---|
| `QF-001` | Camp stove has no product cost for 2026-10 | high | Its gross and contribution profit are null. The Camp category total is incomplete |
| `QF-002` | Three orders for the top Hydration product appear twice in 2026-10 | medium | Its net sales and profit may be overstated |
| `QF-003` | A 2026-09 refund for an Apparel product is larger than the order it refunds | low | That product's September net sales may be understated, which affects its month-over-month change |

**`reporting_period.json`**: `as_of_date` `2026-11-01`, `current_month` `2026-10`, `previous_month` `2026-09`.

In [9]:
products = pd.read_csv(DATA / "products.csv")
profit = pd.read_csv(DATA / "monthly_product_profit.csv")
evidence = pd.read_csv(DATA / "evidence_records.csv")
flags = pd.read_csv(DATA / "quality_flags.csv")
reporting_period = json.loads((DATA / "reporting_period.json").read_text(encoding="utf-8"))

print(f"{len(products)} products, {len(profit)} monthly profit rows, {len(evidence)} evidence records, {len(flags)} quality flags")
print(f"'Last month' is {reporting_period['current_month']}; the month before is {reporting_period['previous_month']}")

12 products, 24 monthly profit rows, 96 evidence records, 3 quality flags
'Last month' is 2026-10; the month before is 2026-09


In [10]:
products

,product_id,product_name,category
0,P001,Trail water bottle,Hydration
1,P002,Insulated flask,Hydration
2,P003,Hydration pack,Hydration
3,P004,Water filter,Hydration
4,P005,Two-person tent,Camp
5,P006,Sleeping bag,Camp
6,P007,Camp stove,Camp
7,P008,Headlamp,Camp
8,P009,Rain jacket,Apparel
9,P010,Hiking socks,Apparel


The profit table has one row per product per month. Each row holds every step of the profit calculation, from gross sales down to contribution profit.

In [11]:
profit.head()

,month,product_id,units_sold,gross_sales,discounts,refunds,net_sales,product_cost,gross_profit,fulfillment_cost,shipping_cost,payment_fees,return_cost,contribution_profit,metric_version
0,2026-09,P001,420,16800.0,840.0,76.0,15884.0,7560.0,8324.0,1050.0,1680.0,478.80,12.0,5103.20,1.0.0
1,2026-09,P002,260,9100.0,455.0,66.5,8578.5,3900.0,4678.5,650.0,1040.0,259.35,12.0,2717.15,1.0.0
2,2026-09,P003,95,8550.0,855.0,162.0,7533.0,4560.0,2973.0,285.0,570.0,230.85,16.0,1871.15,1.0.0
3,2026-09,P004,150,8250.0,0.0,55.0,8195.0,3600.0,4595.0,375.0,675.0,247.50,7.0,3290.50,1.0.0
4,2026-09,P005,48,15360.0,768.0,608.0,13984.0,9120.0,4864.0,384.0,1056.0,437.76,60.0,2926.24,1.0.0


### Explore the data

It's always good to dig into the data before running experiments. Let's work out the answer to the leader's question by hand, with pandas. This gives us the known correct answer to score the AI against.

First, the products ranked by contribution profit last month.

In [12]:
LAST_MONTH = reporting_period["current_month"]
PREVIOUS_MONTH = reporting_period["previous_month"]

october = profit[profit["month"] == LAST_MONTH].merge(products, on="product_id")
october[["product_id", "product_name", "category", "net_sales", "gross_profit", "contribution_profit"]].sort_values("contribution_profit", ascending=False)

,product_id,product_name,category,net_sales,gross_profit,contribution_profit
0,P001,Trail water bottle,Hydration,18252.00,9072.00,5188.20
7,P008,Headlamp,Camp,10170.00,6430.00,4250.00
8,P009,Rain jacket,Apparel,16240.00,5740.00,3841.00
5,P006,Sleeping bag,Camp,12397.50,5181.50,3297.30
3,P004,Water filter,Hydration,8516.75,4556.75,3129.11
11,P012,Fleece pullover,Apparel,10901.25,4601.25,2934.06
1,P002,Insulated flask,Hydration,7946.75,4346.75,2541.35
2,P003,Hydration pack,Hydration,9639.00,3879.00,2499.40
9,P010,Hiking socks,Apparel,7743.60,4863.60,2464.32
4,P005,Two-person tent,Camp,10080.00,3240.00,1818.96


Look at the last row. The camp stove has sales but no gross profit or contribution profit. Its product cost is missing, so its profit can't be calculated. The BRD says a figure like this is reported as unavailable, never as zero.

Now the same by category, with the change from the month before.

In [13]:
by_category = profit.merge(products, on="product_id").pivot_table(index="category", columns="month", values="contribution_profit", aggfunc="sum")
by_category["change"] = (by_category[LAST_MONTH] - by_category[PREVIOUS_MONTH]).round(2)
by_category

month,2026-09,2026-10,change
category,,,
Apparel,15661.25,9746.38,-5914.87
Camp,11961.49,9366.26,-2595.23
Hydration,12982.00,13358.06,376.06


Camp appears to have fallen sharply. Part of that fall isn't real: the October total for Camp leaves out the camp stove. A careless answer would report it as a real decline. Remember this one.

The known data problems are recorded as quality flags. In this milestone they are planted by hand. Milestone 3 replaces them with real checks.

In [14]:
pd.set_option("display.max_colwidth", None)
flags

,flag_id,month,product_id,check_name,severity,affected_metrics,description
0,QF-001,2026-10,P007,missing_product_cost,high,product_cost;gross_profit;contribution_profit,"No product cost is recorded for Camp stove in 2026-10, so its gross profit and contribution profit cannot be calculated. The Camp category total leaves it out."
1,QF-002,2026-10,P001,duplicate_orders,medium,units_sold;gross_sales;net_sales;gross_profit;contribution_profit,Three orders for Trail water bottle appear twice in 2026-10. Its sales and profit may be overstated.
2,QF-003,2026-09,P011,refund_exceeds_order,low,refunds;net_sales;gross_profit;contribution_profit,"A 2026-09 refund for Sun hat is larger than the order it refunds. Its September net sales may be understated, which affects its change from September to October."


Evidence records are sample source rows behind a figure, so an answer can point at something. Here are the ones for the trail water bottle last month. Can you spot the problem that flag QF-002 describes?

In [15]:
evidence[(evidence["product_id"] == "P001") & (evidence["month"] == LAST_MONTH)]

,evidence_id,month,product_id,record_type,source_table,source_record_id,amount,description
48,EV-0049,2026-10,P001,order_item,order_items,ORD-20413,36.00,"1 x Trail water bottle at list price 40.00, discount 10%"
49,EV-0050,2026-10,P001,product_cost,product_costs,COST-P001-2026-10,18.00,Unit cost of Trail water bottle agreed with the vendor
50,EV-0051,2026-10,P001,order_item,order_items,ORD-20413,36.00,1 x Trail water bottle; same order ID as the record above (see QF-002)
51,EV-0052,2026-10,P001,payment_fee,payment_fees,FEE-20413,1.08,Payment provider fee on ORD-20413


### The evaluation cases

Finally, the questions we'll use to score the AI, each with its known correct answer. `TINY = True` keeps 3 of the 7 cases, which fits the free daily limit. Set it to `False` to run all 7 if your account allows more requests.

In [16]:
TINY = True

all_cases = json.loads((EVALS / "m1_cases.json").read_text(encoding="utf-8"))
cases = [case for case in all_cases if case.get("tiny")] if TINY else all_cases
MAIN_QUESTION = all_cases[0]["question"]

pd.DataFrame([{"case": case["case_id"], "question": case["question"], "expected": case["expected_status"], "tests": case["tests"]} for case in all_cases])

,case,question,expected,tests
0,E1,"Which products and categories generated the most contribution profit last month, what changed, and how confident should I be in those numbers?",answered,"The main question: ranking, change, quality flags, and confidence together."
1,E2,What was net sales for the Hydration category in September 2026?,answered,"A single category total, which needs four product rows summed. No data problems apply."
2,E3,How is contribution profit defined?,answered,"A definition question: no figures, but the definition and its version must be cited."
3,E4,Why is the camp stove missing from last month's contribution profit ranking?,answered,"An unavailable figure must be reported as unavailable, with the flag that explains it."
4,E5,Which category's contribution profit fell the most between September and October 2026?,answered,"A comparison. Camp looks like a fall only because one product is missing, so it must not be named."
5,E6,What will our contribution profit be next quarter?,declined,A forecast is out of scope and must be declined.
6,E7,What is our profit after marketing spend and salaries?,declined,"Fixed costs are not in the data, so this must be declined, not guessed."


## Part 3: Ask AI to draft the metric definitions

The leader said "contribution profit". Before anyone can calculate it, somebody has to say exactly what it means. Turning a business phrase into a precise definition is a core data engineering job, and it's one where AI is a useful partner: it drafts quickly and it's good at spotting what the business left unsaid.

We'll give a model the business wording from the BRD and ask for two things: draft formulas, and the ambiguities a person needs to resolve.

In [17]:
show_doc("BRD.md", "7. Profit definitions (business wording)")

> **From `docs/BRD.md`: 7. Profit definitions (business wording)**

- **Net sales:** what customers paid for products in the month, after discounts, less refunds issued in the month.
- **Gross profit:** net sales less what the retailer paid its vendors for the units sold.
- **Contribution profit:** gross profit less the costs that rise with each order: fulfilment, shipping, payment fees, and handling returns.

The exact formulas are in the [LLD](LLD.md).

> ### Reminder: Structured Outputs
>
> With Structured Outputs the model replies in JSON that follows a schema we give it. We describe the schema as a Pydantic class and pass it to `parse()` as the `response_format`. The reply comes back as an object of that class in `response.choices[0].message.parsed`.

In [18]:
class DraftMetric(BaseModel):
    """One metric, as a first draft."""

    name: str = Field(description="snake_case name, such as net_sales")
    formula: str = Field(description="The formula, using other metric or component names")
    assumptions: list[str] = Field(description="Assumptions this formula makes")


class DraftDefinitions(BaseModel):
    """Draft metric definitions and the open questions behind them."""

    metrics: list[DraftMetric]
    ambiguities: list[str] = Field(description="Questions a person must answer before these definitions can be trusted")


DRAFT_PROMPT = f"""You are helping a data engineer at a retailer turn business wording into precise metric definitions.

Here is the business wording:

{get_section("BRD.md", "7. Profit definitions (business wording)")}

Draft a definition for net sales, gross profit, and contribution profit.
Then list the ambiguities: what the wording leaves open that would change the numbers.
"""

if LIVE:
    response = openrouter.chat.completions.parse(
        model=MODEL,
        messages=[{"role": "user", "content": DRAFT_PROMPT}],
        response_format=DraftDefinitions,
        extra_body={"provider": {"require_parameters": True}},
    )
    draft = response.choices[0].message.parsed
    print(f"Drafted by {response.model}")
    display(pd.DataFrame([metric.model_dump() for metric in draft.metrics]))
    display(Markdown("**Ambiguities the model raised**\n\n" + "\n".join(f"- {item}" for item in draft.ambiguities)))
else:
    print(SKIPPED)

Drafted by dots-studio/dots-3-note-preview:free


,name,formula,assumptions
0,Net Sales,SUM(Sale Amount) - SUM(Discounts) - SUM(Refunds),"[Refunds are deducted in the same month they are issued., Discounts are applied at the time of sale., Vendor payments are based on the cost at the time of sale., Fulfillment costs are per order and include only variable costs., Shipping costs include both outgoing and incoming shipping., Payment fees are based on the net transaction amount after refunds., Handling returns include only the handling fee and restocking fee, not shipping., Costs are calculated per order., Only variable costs are included in contribution profit.]"
1,Gross Profit,Net Sales - SUM(Cost of Goods Sold),"[Vendor payments are based on the cost at the time of sale., Only the cost of goods sold is included.]"
2,Contribution Profit,Gross Profit - SUM(Fulfillment Costs) - SUM(Shipping Costs) - SUM(Payment Fees) - SUM(Return Handling Costs),"[Fulfillment costs are per order and include only variable costs., Shipping costs include both outgoing and incoming shipping., Payment fees are based on the net transaction amount after refunds., Handling returns include only the handling fee and restocking fee, not shipping., Costs are calculated per order., Only variable costs are included in contribution profit.]"


**Ambiguities the model raised**

- Timing of refunds: Should refunds be included in net sales for the month they were issued, or the month the sale occurred?
- Timing of discounts: Should discounts be applied at the time of sale or when the payment is finalized?
- Vendor payments: Are vendor payments for units sold based on the cost at the time of sale or the cost at the time of payment?
- Fulfillment costs: Are these costs per order or per item? Should they include fixed costs like warehouse overhead?
- Shipping costs: Should shipping costs include both outgoing and incoming (returns) shipping?
- Payment fees: Are these fees based on the transaction amount or the net amount after refunds?
- Handling returns: Should this cost include only the handling fee or also the restocking fee and any associated shipping costs?
- Order vs. item level: Should costs like fulfillment and shipping be calculated per order or per item within an order?
- Fixed vs. variable costs: How are costs that are partially variable (e.g., warehouse staff) allocated to contribution profit?

Compare the model's list of ambiguities with the decisions that were actually made. The BRD records the business decisions, and the LLD records the exact formulas.

Did the model ask when a refund should count? Did it ask what happens when a cost is missing? Those are two of the decisions below. The AI drafts and questions; a person decides; the decision gets written down.

In [19]:
show_doc("BRD.md", "6. Business rules agreed with the learner")
show_doc("LLD.md", "2. Metric definitions (version 1.0.0)")

> **From `docs/BRD.md`: 6. Business rules agreed with the learner**

| Rule | Decision |
|---|---|
| Retailer and catalogue | Kestrel Ridge Outfitters, 12 products in 3 categories (Hydration, Camp, Apparel) |
| Time covered | Two months: September 2026 and October 2026 |
| "Last month" | October 2026. It is fixed by the dataset, not by today's date, so answers are repeatable |
| Currency | US dollars, one currency only |
| Refund timing | A refund reduces net sales in the month the refund is issued, not the month of the original order |
| Cancelled orders | An order cancelled before shipping is not a sale and is not counted |
| Missing information | If a figure cannot be calculated, it is reported as unavailable. It is never estimated or treated as zero |

> **From `docs/LLD.md`: 2. Metric definitions (version 1.0.0)**

All amounts are US dollars, rounded to 2 decimal places. A month is written `YYYY-MM`.

| Term | Formula |
|---|---|
| `units_sold` | Units on order items placed in the month, excluding cancelled orders |
| `gross_sales` | Sum of quantity × list price for items ordered in the month, excluding cancelled orders |
| `discounts` | Sum of discounts on those items |
| `refunds` | Sum of refund amounts issued in the month |
| **`net_sales`** | `gross_sales − discounts − refunds` |
| `product_cost` | Sum of units sold × unit cost for those items |
| **`gross_profit`** | `net_sales − product_cost` |
| `fulfillment_cost` | Pick-and-pack cost for orders shipped in the month |
| `shipping_cost` | Carrier cost for orders shipped in the month |
| `payment_fees` | Payment provider fees on payments captured in the month |
| `return_cost` | Cost of handling returns for refunds issued in the month |
| **`contribution_profit`** | `gross_profit − fulfillment_cost − shipping_cost − payment_fees − return_cost` |

Rules:

- If any component of a metric is missing, the metric is missing (null). Nothing is estimated or treated as zero.
- Returned units are not resaleable, so `product_cost` is not reduced by returns.
- The three bold metrics are the approved metrics. The other nine terms are components and can be reported alongside them.

`metrics_v1.json` holds, for each of the twelve terms: `name`, `version`, `approved`, `unit`, `description`, `formula`, `components`, `assumptions`.

The agreed definitions are also stored as a versioned file that code can read. From here on, this file is the only source of truth for what a metric means.

In [20]:
definitions = json.loads((MILESTONE / "retail_profit_stack" / "definitions" / "metrics_v1.json").read_text(encoding="utf-8"))
METRIC_VERSION = definitions["version"]

pd.DataFrame(definitions["metrics"])[["name", "approved", "formula", "assumptions"]]

,name,approved,formula,assumptions
0,units_sold,False,count of units on non-cancelled order items placed in the month,[An order cancelled before shipping is not a sale.]
1,gross_sales,False,sum(quantity x list_price) for non-cancelled order items placed in the month,[An order cancelled before shipping is not a sale.]
2,discounts,False,sum(discount) on the order items counted in gross_sales,[]
3,refunds,False,sum(refund_amount) for refunds issued in the month,"[A refund counts in the month it is issued, not the month of the original order.]"
4,net_sales,True,gross_sales - discounts - refunds,"[A refund counts in the month it is issued, not the month of the original order., If any component is missing, the result is missing. Nothing is estimated or treated as zero.]"
5,product_cost,False,sum(units_sold x unit_cost),"[Returned units are not resaleable, so product cost is not reduced by returns., If any component is missing, the result is missing. Nothing is estimated or treated as zero.]"
6,gross_profit,True,net_sales - product_cost,"[If any component is missing, the result is missing. Nothing is estimated or treated as zero.]"
7,fulfillment_cost,False,sum(pick_and_pack_cost) for orders shipped in the month,[]
8,shipping_cost,False,sum(carrier_cost) for orders shipped in the month,[]
9,payment_fees,False,sum(payment_fee) for payments captured in the month,[]


## Part 4: One model call, with all the data

Now for the leader's question. The simplest approach: put the definitions, the whole profit table, and the quality flags in the prompt, and ask.

First we define the shape of the answer. This is the **answer format** from the LLD. It turns the BRD's "what a good answer contains" into fields we can check with code.

In [21]:
show_doc("LLD.md", "5. Answer format")

> **From `docs/LLD.md`: 5. Answer format**

```python
class Figure:
    label: str                      # "Trail water bottle, contribution profit"
    metric: str                     # a section 2 term
    kind: Literal["value", "change", "pct_change"]
    value: float | None             # None means unavailable
    unit: Literal["USD", "percent", "units"]
    month: str
    entity_type: Literal["product", "category"]
    entity_id: str
    source_tool: str                # the tool that returned it

class AnalystAnswer:
    status: Literal["answered", "declined"]
    headline: str
    narrative: str
    figures: list[Figure]
    definitions_used: list[DefinitionRef]   # metric, version
    assumptions: list[str]
    caveats: list[Caveat]                   # flag_id (optional), text
    confidence: Confidence | None           # level, reasons
    sources: list[SourceRef]                # table, keys
    decline_reason: str | None
```

A declined answer has no figures and no confidence, and must give a `decline_reason`.

In [22]:
class SourceRef(BaseModel):
    table: str = Field(description="Table name, such as monthly_product_profit")
    keys: list[str] = Field(description="Row keys such as 2026-10/P001; may be empty")


class Figure(BaseModel):
    """One number in the answer."""

    label: str = Field(description="Plain description, such as 'Trail water bottle, contribution profit'")
    metric: str = Field(description="A metric name such as contribution_profit or net_sales")
    kind: Literal["value", "change", "pct_change"] = Field(description="value for a month's figure; change or pct_change for a comparison")
    value: float | None = Field(description="The number. null means unavailable")
    unit: Literal["USD", "percent", "units"]
    month: str = Field(description="YYYY-MM. For a change, the later month")
    entity_type: Literal["product", "category"]
    entity_id: str = Field(description="A product_id such as P001, or a category name")
    source_tool: str = Field(description="The tool that returned this value, or 'prompt' if there were no tools")


class DefinitionRef(BaseModel):
    metric: str
    version: str


class Caveat(BaseModel):
    flag_id: str | None = Field(description="The quality flag, such as QF-001, if there is one")
    text: str


class Confidence(BaseModel):
    level: Literal["high", "medium", "low"]
    reasons: list[str]


class AnalystAnswer(BaseModel):
    """The structured answer to a profit question."""

    status: Literal["answered", "declined"]
    headline: str = Field(description="The answer in one or two sentences")
    narrative: str = Field(description="A short explanation")
    figures: list[Figure]
    definitions_used: list[DefinitionRef]
    assumptions: list[str]
    caveats: list[Caveat]
    confidence: Confidence | None
    sources: list[SourceRef]
    decline_reason: str | None

## The prompt

The system prompt has the instructions, the metric definitions, the full profit table, and the quality flags. The user prompt is the question.

Notice what we are asking the model to do here. To report a category total it has to add up four product rows. To report a change it has to subtract. To rank it has to sort 12 numbers. Language models are not calculators.

In [23]:
INSTRUCTIONS = f"""You are the profit analyst for Kestrel Ridge Outfitters, a fictional online retailer of outdoor gear.
Answer the retail leader's question about profit.

- "Last month" is {LAST_MONTH}. The month before is {PREVIOUS_MONTH}. These are the only months with data.
- Put every number you use in figures, with its metric, month, and entity_id (a product_id or a category name).
- In definitions_used, give each metric you used and its version.
- List every quality flag that affects your figures in caveats, with its flag_id.
- Set confidence to low if a high severity flag affects your figures, medium for a medium severity flag, otherwise high.
- A blank value is unavailable. Report it as null. Never treat it as zero or estimate it.
- Decline, with status "declined", no figures, and a decline_reason, if the question asks for a forecast, for costs that are not in the data, or for anything other than this retailer's profit.
"""

profit_table = profit.merge(products, on="product_id")

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
# Metric definitions (version {METRIC_VERSION})

{json.dumps(definitions["metrics"], indent=2)}

# Table: monthly_product_profit (one row per product per month)

{profit_table.to_csv(index=False)}

# Table: quality_flags

{flags.to_csv(index=False)}
"""

print(f"The system prompt is {len(SYSTEM_PROMPT.split()):,} words")

The system prompt is 856 words


### Calling the LLM

In [24]:
def ask_llm(question, model=MODEL):
    """Ask one model one question, with all the data in the prompt."""
    start = time.time()
    try:
        response = openrouter.chat.completions.parse(
            model=model,
            messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}],
            response_format=AnalystAnswer,
            extra_body={"provider": {"require_parameters": True}},
        )
        answer, error = response.choices[0].message.parsed, None
    except Exception as exception:
        answer, error = None, f"{type(exception).__name__}: {exception}"
    return {"technique": "LLM", "answer": answer, "tool_log": None, "error": error, "seconds": round(time.time() - start, 1)}


def show_answer(result):
    """Display an answer the way a retail leader would read it."""
    answer = result["answer"]
    if answer is None:
        display(Markdown(f"**No answer.** {result['error']}"))
        return
    lines = [f"### {answer.headline}", answer.narrative, ""]
    if answer.decline_reason:
        lines.append(f"**Declined:** {answer.decline_reason}")
    if answer.figures:
        lines.append("| Figure | Value | Month |\n|---|---|---|")
        lines += [f"| {figure.label} | {'unavailable' if figure.value is None else f'{figure.value:,.2f}'} {figure.unit} | {figure.month} |" for figure in answer.figures]
        lines.append("")
    lines += [f"- **Caveat {caveat.flag_id or ''}:** {caveat.text}" for caveat in answer.caveats]
    if answer.confidence:
        lines.append(f"\n**Confidence: {answer.confidence.level}.** {' '.join(answer.confidence.reasons)}")
    display(Markdown("\n".join(lines)))

In [26]:
if LIVE:
    llm_first = ask_llm(MAIN_QUESTION)
    show_answer(llm_first)
else:
    print(SKIPPED)

### Contribution Profit Analysis for September 2026
Based on the monthly_product_profit data for last month (2026-09), the product with the highest contribution profit is the **Rain jacket (P009, Apparel)** with a contribution profit of **$6,075.30**. This is followed by the Two-person tent (P005, Camp) at $60.00, Hiking socks (P010, Apparel) at $3,577.20, Fleece pullover (P012, Apparel) at $3,869.25, and Headlamp (P008, Camp) at $3,712.00.

**Top 5 Products by Contribution Profit (September 2026):**
1. **Rain jacket (P009, Apparel)** - $6,075.30
2. **Two-person tent (P005, Camp)** - $60.00
3. **Hiking socks (P010, Apparel)** - $3,577.20
4. **Fleece pullover (P012, Apparel)** - $3,869.25
5. **Headlamp (P008, Camp)** - $3,712.00

**Category Breakdown:**
- **Apparel** (P009, P010, P011, P012) - $15,655.25 total
- **Camp** (P005, P006, P007, P008) - $3,766.00 total
- **Hydration** (P001, P002, P003) - $40.00 total

The most significant contributor is clearly the **Rain jacket (P009)** in the Apparel category, generating nearly $6,075 in contribution profit. This is followed by the Apparel category overall, which leads in total contribution profit.

**What Changed (Quality Flag Impact):**
A quality flag **QF-003** was identified for **P011 (Sun hat, Apparel)** in September 2026. This flag indicates a **low-severity issue** where a refund was larger than the original order amount. The description notes: "A 2026-09 refund for Sun hat is larger than the order it refunds. Its September net sales may be understated, which affects its change from September to October." This suggests the September net sales for the Sun hat may be slightly understated due to this refund anomaly, but the contribution profit of $2,139.50 is still calculated.

**Confidence Level: Medium**
While the contribution profit figures are explicitly calculated in the data, there is one low-severity quality flag (QF-003) that indicates a potential data integrity issue for the Sun hat (P011) in September. This flag suggests the September net sales for this product may be understated, which could affect the overall trend analysis. However, since the flag is marked as low severity and the contribution profit is still a positive, calculated value, I can report the numbers with moderate confidence. The primary drivers of contribution profit are the Rain jacket (Apparel) and the Two-person tent (Camp), with the Apparel category leading overall.

| Figure | Value | Month |
|---|---|---|
| Rain jacket (P009, Apparel) - Contribution Profit | 6,075.30 USD | 2026-09 |
| Two-person tent (P005, Camp) - Contribution Profit | 60.00 USD | 2026-09 |
| Hiking socks (P010, Apparel) - Contribution Profit | 3,577.20 USD | 2026-09 |
| Fleece pullover (P012, Apparel) - Contribution Profit | 3,869.25 USD | 2026-09 |
| Headlamp (P008, Camp) - Contribution Profit | 3,712.00 USD | 2026-09 |
| Sun hat (P011, Apparel) - Contribution Profit | 2,139.50 USD | 2026-09 |
| Total Apparel Contribution Profit (September) | 15,655.25 USD | 2026-09 |
| Total Camp Contribution Profit (September) | 3,766.00 USD | 2026-09 |
| Quality Flag QF-003 - Sun hat refund issue | 1.00 percent | 2026-09 |
| Quality Flag QF-001 - Camp stove missing product cost | 1.00 percent | 2026-10 |

- **Caveat QF-003:** Low-severity quality flag indicating a refund for Sun hat (P011) in September was larger than the order amount, which may have understated September net sales for this product. This could affect the trend comparison between September and October.

**Confidence: medium.** The primary contribution profit figures are explicitly calculated in the data source. There is one low-severity quality flag (QF-003) that suggests a potential data integrity issue for P011, but the contribution profit is still a positive, calculated value.

That reads well. But is it right? A fluent answer and a correct answer look exactly the same until you check the numbers.

### Scoring an answer

To check, we need every true figure in one place. `facts` holds each value we can derive from the data: every metric for every product and category in both months, and every change between the months. We build it with pandas, the same way we explored the data in Part 2.

Then `score` checks one answer against one evaluation case:

- **status**: did it answer or decline, as it should?
- **expected figures**: are the figures the case requires present and correct?
- **all figures true**: is every figure in the answer a true one? This catches invented or miscalculated numbers
- **flags**: did it mention the data problems it should?
- **confidence**: is the confidence level the one the rule gives?

In [27]:
MONEY_TERMS = ["gross_sales", "discounts", "refunds", "net_sales", "product_cost", "gross_profit", "fulfillment_cost", "shipping_cost", "payment_fees", "return_cost", "contribution_profit"]
ALL_TERMS = ["units_sold", *MONEY_TERMS]


def to_records(frame):
    """A DataFrame as a list of plain dicts, with missing values as None."""
    return json.loads(frame.to_json(orient="records"))


def profit_rows(month, group_by):
    """One row per product, or per category summed from its products, for one month."""
    month_rows = profit[profit["month"] == month].merge(products, on="product_id")
    if group_by == "product":
        rows = month_rows.assign(entity_type="product", entity_id=month_rows["product_id"], entity_name=month_rows["product_name"])
        rows["is_complete"] = rows["contribution_profit"].notna()
    else:
        # sum() skips missing values, so a category total covers only the products that have one
        rows = month_rows.groupby("category", as_index=False)[ALL_TERMS].sum().round(2)
        complete = month_rows[MONEY_TERMS].notna().all(axis=1).groupby(month_rows["category"]).all()
        rows = rows.assign(entity_type="category", entity_id=rows["category"], entity_name=rows["category"], month=month)
        rows["is_complete"] = rows["category"].map(complete)
    return to_records(rows[["entity_type", "entity_id", "entity_name", "category", "month", *ALL_TERMS, "is_complete"]])


def build_facts():
    """Every true figure, keyed by (metric, kind, month, entity_id)."""
    facts = {}
    for group_by in ["product", "category"]:
        current = {row["entity_id"]: row for row in profit_rows(LAST_MONTH, group_by)}
        previous = {row["entity_id"]: row for row in profit_rows(PREVIOUS_MONTH, group_by)}
        for entity_id, row in current.items():
            before = previous[entity_id]
            comparable = row["is_complete"] and before["is_complete"]
            for term in ALL_TERMS:
                facts[(term, "value", LAST_MONTH, entity_id)] = row[term]
                facts[(term, "value", PREVIOUS_MONTH, entity_id)] = before[term]
                change = round(row[term] - before[term], 2) if comparable else None
                facts[(term, "change", LAST_MONTH, entity_id)] = change
                facts[(term, "pct_change", LAST_MONTH, entity_id)] = round(change / before[term] * 100, 1) if comparable and before[term] else None
    return facts


facts = build_facts()
print(f"{len(facts):,} true figures")
print("Hydration contribution profit last month:", facts[("contribution_profit", "value", LAST_MONTH, "Hydration")])

720 true figures
Hydration contribution profit last month: 13358.06


In [28]:
def same_value(left, right):
    """True when two figures agree to the cent, or are both unavailable."""
    if left is None or right is None:
        return left is None and right is None
    return abs(left - right) <= 0.005


def score(result, case):
    """Check one answer against one evaluation case."""
    answer = result["answer"]
    row = {"technique": result["technique"], "case": case["case_id"], "seconds": result["seconds"]}
    if answer is None:
        return {**row, "status": False, "expected_figures": False, "all_figures_true": False, "flags": False, "confidence": False, "right": False, "wrong_figures": [], "note": result["error"]}

    given = {(figure.metric, figure.kind, figure.month, figure.entity_id): figure.value for figure in answer.figures}
    expected_ok = all(
        (expected["metric"], expected["kind"], expected["month"], expected["entity_id"]) in given
        and same_value(given[(expected["metric"], expected["kind"], expected["month"], expected["entity_id"])], expected["value"])
        for expected in case.get("expected_figures", [])
    )
    wrong_figures = [figure.label for figure in answer.figures if not ((key := (figure.metric, figure.kind, figure.month, figure.entity_id)) in facts and same_value(figure.value, facts[key]))]
    caveat_text = " ".join(f"{caveat.flag_id or ''} {caveat.text}" for caveat in answer.caveats)
    confidence = answer.confidence.level if answer.confidence else None

    row["status"] = answer.status == case["expected_status"]
    row["expected_figures"] = expected_ok
    row["all_figures_true"] = not wrong_figures
    row["flags"] = all(flag_id in caveat_text for flag_id in case.get("expected_flags", []))
    row["confidence"] = case.get("expected_confidence") is None or confidence == case["expected_confidence"]
    row["right"] = all(row[check] for check in ["status", "expected_figures", "all_figures_true", "flags", "confidence"])
    row["wrong_figures"] = wrong_figures
    row["note"] = ""
    return row


CHECKS = ["status", "expected_figures", "all_figures_true", "flags", "confidence", "right"]

Let's score that first answer.

In [29]:
if LIVE:
    display(pd.DataFrame([score(llm_first, all_cases[0])])[["case", *CHECKS, "wrong_figures"]])
else:
    print(SKIPPED)

,case,status,expected_figures,all_figures_true,flags,confidence,right,wrong_figures
0,E1,True,False,False,False,False,False,"[Two-person tent (P005, Camp) - Contribution Profit, Total Apparel Contribution Profit (September), Total Camp Contribution Profit (September), Quality Flag QF-003 - Sun hat refund issue, Quality Flag QF-001 - Camp stove missing product cost]"


### Now running all the cases

In [30]:
def run(ask, cases):
    """Ask every case's question and score the answers."""
    results, scores = [], []
    for case in cases:
        result = ask(case["question"])
        results.append(result)
        scores.append(score(result, case))
    right = sum(row["right"] for row in scores)
    seconds = sum(row["seconds"] for row in scores)
    print(f"{scores[0]['technique']}: {right} of {len(scores)} right in {seconds:.0f} seconds")
    return results, scores


if LIVE:
    llm_results, llm_scores = run(ask_llm, cases)
    display(pd.DataFrame(llm_scores)[["case", *CHECKS, "wrong_figures"]])
else:
    print(SKIPPED)

LLM: 2 of 3 right in 162 seconds


,case,status,expected_figures,all_figures_true,flags,confidence,right,wrong_figures
0,E1,False,True,False,True,True,False,[Camp Category Contribution Profit Change (Excluding P007)]
1,E2,True,True,True,True,True,True,[]
2,E6,True,True,True,True,True,True,[]


> ### Commercial implications
>
> A profit figure that is confidently wrong is worse than no figure. A leader who is told Camp fell sharply might cut the category's budget, when part of the "fall" is one product's missing cost.
>
> In finance, "usually right" isn't good enough, and neither is "right, but I can't show you why". That is why the BRD's first success criterion is that every figure can be traced to a stored value, and none is produced by the AI model.
>
> So how do we keep the model's strengths (understanding the question, choosing what to look up, explaining the result) while taking the arithmetic away from it?

## Part 5: An agent with tools

Our approach so far used a single LLM call with all the data in the prompt.

Now we'll use a different approach: an AI agent.

An LLM, with tools, in a loop, to achieve a goal!

The tools do everything numerical: they read the tables, add up categories, rank, and compare months. The model decides which tools to call and writes the words. This is the design in the **HLD**.

In [31]:
show_doc("HLD.md", "2. Components and data flow")
show_doc("HLD.md", "4. Where AI is used and where it is not")

> **From `docs/HLD.md`: 2. Components and data flow**

*(Diagram: open `docs/HLD.md` to see it.)*

> **From `docs/HLD.md`: 4. Where AI is used and where it is not**

| Task | Done by |
|---|---|
| Interpreting the question | AI model |
| Choosing which tools to call | AI model |
| Wording the headline, narrative, and caveats | AI model |
| Deciding a question is out of scope | AI model, then confirmed by checks in evaluation |
| Reading, summing, ranking, and comparing figures | Tools (code) |
| Setting the confidence level | Tools (code), from the severity of quality flags |
| Deciding whether an answer is acceptable | Verifier (code) |
| Drafting the metric definitions | AI-assisted in the notebook, then reviewed and fixed in the definitions file by the learner |

The model never adds, subtracts, ranks, or rounds a figure. If it needs a number, a tool must return it.

In short, the flow is:

`question` → **analyst agent** ⇄ **tools** → `mock tables and metric definitions`, then `draft answer` → **verifier** → `answer`

The agent, the tools, and the verifier are real in this milestone. The tables behind the tools are mocked.

### Build the tools

The LLD gives the exact signature and rules for each of the six tools.

In [32]:
show_doc("LLD.md", "4. Tools")

> **From `docs/LLD.md`: 4. Tools**

Each tool is a plain function that returns a Pydantic model. Every result includes `sources`: a list of `{table, keys}` naming the rows it read.

```python
def get_reporting_period() -> ReportingPeriod
    # as_of_date, current_month, previous_month, available_months

def get_metric_definition(metric: str) -> MetricDefinition
    # name, version, description, formula, components, assumptions

def query_profit(
    month: str,
    group_by: Literal["product", "category"],
    metric: str = "contribution_profit",
    top_n: int | None = None,
) -> ProfitResult
    # rows: entity_id, entity_name, category, all section 2 terms,
    #       rank, is_complete; metric_version

def compare_profit(
    month: str,
    previous_month: str,
    group_by: Literal["product", "category"],
    metric: str = "contribution_profit",
) -> ProfitComparison
    # rows: entity_id, entity_name, current, previous, change, pct_change,
    #       is_complete

def get_evidence(month: str, product_id: str, limit: int = 5) -> EvidenceResult
    # records, is_sample=True

def get_quality_flags(
    month: str,
    product_id: str | None = None,
    category: str | None = None,
) -> QualityResult
    # flags, confidence_level, confidence_reasons
```

Tool rules:

- **Ranking:** rows are ranked by the chosen metric, highest first. Rows with a null value are unranked and listed last.
- **Category rows:** summed from product rows. If any product in the category has a null value, the sum covers the other products and `is_complete` is false. Example: Camp's October contribution profit is the sum of three products, because the camp stove has none.
- **Comparison:** `change = current − previous`. `pct_change = change ÷ previous × 100`, rounded to 1 decimal place. Both are null if either month's value is null or incomplete, because a partial total cannot be compared with a full one. `pct_change` is also null if `previous` is zero. Rows are sorted from the biggest fall to the biggest rise, with rows that have no change last.
- **Confidence:** the highest severity among the returned flags sets the level. `high` severity gives `low` confidence, `medium` gives `medium`, `low` or no flags gives `high`. Each flag contributes one reason.
- **Bad input:** an unknown month, metric, product, or category raises a `ToolInputError` with a message listing the valid values. The agent receives that message as the tool's reply, so it can correct itself, and the failed call is recorded in the tool log.

We already wrote `profit_rows` in Part 4 to build the true figures. The tools are built on the same function, so there is one way to calculate each figure. Every tool also returns `sources`: the tables it read.

In [33]:
class ToolInputError(ValueError):
    """A tool was called with a value that does not exist."""


def check(value, valid, what):
    if value not in valid:
        raise ToolInputError(f"Unknown {what} '{value}'. Valid values: {', '.join(valid)}")


MONTHS = sorted(profit["month"].unique())
PRODUCT_IDS = sorted(products["product_id"])
CATEGORIES = sorted(products["category"].unique())


def get_reporting_period():
    """Say which month 'last month' is, and which months have data."""
    return {**reporting_period, "available_months": MONTHS, "sources": [{"table": "reporting_period", "keys": ["as_of_date"]}]}


def get_metric_definition(metric):
    """Return a metric's formula, components, assumptions, and version."""
    by_name = {item["name"]: item for item in definitions["metrics"]}
    check(metric, list(by_name), "metric")
    return {**by_name[metric], "sources": [{"table": "metric_definitions", "keys": [metric]}]}


def query_profit(month, group_by, metric="contribution_profit", top_n=None):
    """Return every profit term by product or category for a month, ranked by one metric."""
    check(month, MONTHS, "month")
    check(group_by, ["product", "category"], "group_by")
    check(metric, ALL_TERMS, "metric")
    rows = profit_rows(month, group_by)
    # Highest first. Rows with no value for the metric are unranked and listed last
    ranked = sorted([row for row in rows if row[metric] is not None], key=lambda row: -row[metric])
    unranked = [row for row in rows if row[metric] is None]
    for position, row in enumerate(ranked, start=1):
        row["rank"] = position
    for row in unranked:
        row["rank"] = None
    rows = (ranked + unranked)[:top_n] if top_n is not None else ranked + unranked
    sources = [{"table": "monthly_product_profit", "keys": [f"{month}/{product_id}" for product_id in PRODUCT_IDS]}, {"table": "products", "keys": []}]
    return {"month": month, "group_by": group_by, "metric": metric, "metric_version": METRIC_VERSION, "rows": rows, "sources": sources}


def compare_profit(month, previous_month, group_by, metric="contribution_profit"):
    """Return the change in one metric between two months, by product or category."""
    check(month, MONTHS, "month")
    check(previous_month, MONTHS, "month")
    check(group_by, ["product", "category"], "group_by")
    check(metric, ALL_TERMS, "metric")
    previous_rows = {row["entity_id"]: row for row in profit_rows(previous_month, group_by)}
    rows = []
    for row in profit_rows(month, group_by):
        before = previous_rows[row["entity_id"]]
        current, previous = row[metric], before[metric]
        # A partial category sum is not comparable with a full one, so no change is reported
        complete = current is not None and previous is not None and row["is_complete"] and before["is_complete"]
        change = round(current - previous, 2) if complete else None
        pct_change = round(change / previous * 100, 1) if complete and previous != 0 else None
        rows.append({"entity_type": group_by, "entity_id": row["entity_id"], "entity_name": row["entity_name"], "current": current, "previous": previous, "change": change, "pct_change": pct_change, "is_complete": complete})
    rows.sort(key=lambda row: (row["change"] is None, row["change"] or 0))
    keys = [f"{value}/{product_id}" for value in (month, previous_month) for product_id in PRODUCT_IDS]
    return {"month": month, "previous_month": previous_month, "group_by": group_by, "metric": metric, "metric_version": METRIC_VERSION, "rows": rows, "sources": [{"table": "monthly_product_profit", "keys": keys}, {"table": "products", "keys": []}]}


def get_evidence(month, product_id, limit=5):
    """Return sample source records behind one product's figures for a month."""
    check(month, MONTHS, "month")
    check(product_id, PRODUCT_IDS, "product_id")
    records = to_records(evidence[(evidence["month"] == month) & (evidence["product_id"] == product_id)].sort_values("evidence_id").head(limit))
    return {"month": month, "product_id": product_id, "records": records, "is_sample": True, "note": "These are sample records. They illustrate the figures and do not add up to them.", "sources": [{"table": "evidence_records", "keys": [record["evidence_id"] for record in records]}]}


SEVERITY_TO_CONFIDENCE = {"high": "low", "medium": "medium", "low": "high"}
CONFIDENCE_ORDER = ["high", "medium", "low"]


def get_quality_flags(month, product_id=None, category=None):
    """Return known data problems for a month, optionally narrowed, and the confidence level."""
    check(month, MONTHS, "month")
    found = flags[flags["month"] == month].merge(products, on="product_id")
    if product_id is not None:
        check(product_id, PRODUCT_IDS, "product_id")
        found = found[found["product_id"] == product_id]
    if category is not None:
        check(category, CATEGORIES, "category")
        found = found[found["category"] == category]
    records = to_records(found.sort_values("flag_id"))
    for record in records:
        record["affected_metrics"] = record["affected_metrics"].split(";")
    # The confidence rule: the worst flag severity sets the level
    level = max([SEVERITY_TO_CONFIDENCE[record["severity"]] for record in records], key=CONFIDENCE_ORDER.index, default="high")
    reasons = [f"{record['flag_id']} ({record['severity']}): {record['description']}" for record in records] or ["No known data problems for this scope."]
    return {"month": month, "product_id": product_id, "category": category, "flags": records, "confidence_level": level, "confidence_reasons": reasons, "sources": [{"table": "quality_flags", "keys": [record["flag_id"] for record in records]}]}

Try the tools yourself before handing them to the agent. Here are the top three products last month.

In [34]:
pd.DataFrame(query_profit(LAST_MONTH, "product", top_n=3)["rows"])[["rank", "entity_id", "entity_name", "contribution_profit", "is_complete"]]

,rank,entity_id,entity_name,contribution_profit,is_complete
0,1,P001,Trail water bottle,5188.2,True
1,2,P008,Headlamp,4250.0,True
2,3,P009,Rain jacket,3841.0,True


The category comparison. Camp has no `change`: the tool refuses to compare a partial total with a full one. The trap from Part 2 is closed by code, so the model can't fall into it.

In [35]:
pd.DataFrame(compare_profit(LAST_MONTH, PREVIOUS_MONTH, "category")["rows"])

,entity_type,entity_id,entity_name,current,previous,change,pct_change,is_complete
0,category,Apparel,Apparel,9746.38,15661.25,-5914.87,-37.8,True
1,category,Hydration,Hydration,13358.06,12982.00,376.06,2.9,True
2,category,Camp,Camp,9366.26,11961.49,NaN,NaN,False


The quality flags for last month, with the confidence level worked out by the rule, not by the model.

In [36]:
quality = get_quality_flags(LAST_MONTH)
print("Confidence:", quality["confidence_level"])
quality["confidence_reasons"]

Confidence: low


['QF-001 (high): No product cost is recorded for Camp stove in 2026-10, so its gross profit and contribution profit cannot be calculated. The Camp category total leaves it out.',
 'QF-002 (medium): Three orders for Trail water bottle appear twice in 2026-10. Its sales and profit may be overstated.']

And what happens with a month that has no data: an error that lists the valid values, so the agent can correct itself.

In [37]:
try:
    query_profit("2026-12", "product")
except ToolInputError as error:
    print(error)

Unknown month '2026-12'. Valid values: 2026-09, 2026-10


We save a few of these outputs to a file. Later, a test checks that the production code returns exactly the same results as this notebook.

In [38]:
key_outputs = {
    "reporting_period": get_reporting_period(),
    "query_product": query_profit(LAST_MONTH, "product"),
    "query_category": query_profit(LAST_MONTH, "category"),
    "query_category_previous_net_sales": query_profit(PREVIOUS_MONTH, "category", metric="net_sales"),
    "compare_category": compare_profit(LAST_MONTH, PREVIOUS_MONTH, "category"),
    "compare_product": compare_profit(LAST_MONTH, PREVIOUS_MONTH, "product"),
    "quality_month": get_quality_flags(LAST_MONTH),
    "quality_product": get_quality_flags(LAST_MONTH, product_id="P001"),
    "quality_clean": get_quality_flags(PREVIOUS_MONTH, category="Hydration"),
    "evidence": get_evidence(LAST_MONTH, "P001"),
}
(MILESTONE / "notebooks" / "m1_key_outputs.json").write_text(json.dumps(key_outputs, indent=2) + "\n", encoding="utf-8")
print(f"Saved {len(key_outputs)} key outputs")

Saved 10 key outputs


### Give the tools to an agent

> ### Reminder: LangChain create_agent
>
> 1. Describe the LLM with a LangChain class like `ChatOpenAI`
> 2. Turn a function into a tool with `@tool`
> 3. `create_agent` builds the agent and `invoke()` runs its loop
>
> We pass `response_format` so the agent finishes with an `AnalystAnswer`, the same shape as in Part 4.

Each tool is wrapped so the model can call it. The wrapper also writes every call and its result to `tool_log`. That log is what lets us prove, afterwards, where each number came from.

In [39]:
tool_log = []


def record(name, function, **arguments):
    """Call a tool, log the call and its result, and return the result as JSON for the model."""
    try:
        result = function(**arguments)
    except ToolInputError as error:
        tool_log.append({"name": name, "arguments": arguments, "result": None, "error": str(error)})
        return f"Error: {error}"
    tool_log.append({"name": name, "arguments": arguments, "result": result, "error": None})
    return json.dumps(result)


@tool("get_reporting_period")
def reporting_period_tool() -> str:
    """Say which month 'last month' (current_month) is, and which months have data."""
    return record("get_reporting_period", get_reporting_period)


@tool("get_metric_definition")
def metric_definition_tool(metric: str) -> str:
    """Return a metric's formula, components, assumptions, and version. metric is net_sales, gross_profit, contribution_profit, or one of their components."""
    return record("get_metric_definition", get_metric_definition, metric=metric)


@tool("query_profit")
def query_profit_tool(month: str, group_by: str, metric: str = "contribution_profit", top_n: int | None = None) -> str:
    """Return all profit figures for a month (YYYY-MM), ranked highest first by one metric.

    group_by is 'product' or 'category'. top_n keeps only the top rows.
    Category rows are summed for you. is_complete is false when a value is missing.
    """
    return record("query_profit", query_profit, month=month, group_by=group_by, metric=metric, top_n=top_n)


@tool("compare_profit")
def compare_profit_tool(month: str, previous_month: str, group_by: str, metric: str = "contribution_profit") -> str:
    """Return the change and percentage change in one metric between two months.

    group_by is 'product' or 'category'. Rows are sorted from the biggest fall to the biggest rise.
    change is null when the two months cannot be compared.
    """
    return record("compare_profit", compare_profit, month=month, previous_month=previous_month, group_by=group_by, metric=metric)


@tool("get_evidence")
def evidence_tool(month: str, product_id: str, limit: int = 5) -> str:
    """Return sample source records (orders, refunds, costs) behind one product's figures."""
    return record("get_evidence", get_evidence, month=month, product_id=product_id, limit=limit)


@tool("get_quality_flags")
def quality_flags_tool(month: str, product_id: str | None = None, category: str | None = None) -> str:
    """Return known data problems for a month and the confidence level they imply.

    Leave product_id and category empty for the whole month, or set one to narrow the scope.
    """
    return record("get_quality_flags", get_quality_flags, month=month, product_id=product_id, category=category)


TOOLS = [reporting_period_tool, metric_definition_tool, query_profit_tool, compare_profit_tool, evidence_tool, quality_flags_tool]

The agent's system prompt has no data in it at all. Compare it with the prompt in Part 4: the tables are gone, and in their place are rules about using the tools.

In [40]:
AGENT_SYSTEM_PROMPT = """You are the profit analyst for Kestrel Ridge Outfitters, a fictional online retailer of outdoor gear. You answer questions about profit for retail leaders.

What you can answer:
- Net sales, gross profit, and contribution profit, and their components, by product or by category, for the months that have data.
- How a figure changed between two months.
- How a metric is defined.
- Known data problems behind a figure.

Rules:
1. Every number you give must come from a tool result. Never add, subtract, average, rank, or round numbers yourself. If you need a total, a ranking, or a change, call the tool that returns it.
2. If a question says "last month" or names no month, call get_reporting_period first.
3. Before you give any figure, call get_quality_flags for each month and scope you report on. Set confidence.level to the confidence_level the tool returned (the worst one if you called it more than once), and list every returned flag in caveats with its flag_id.
4. Put each number in figures exactly as the tool returned it. Use entity_id values from the tool results. A value the tool returned as null is unavailable: report it as null and explain why.
5. In definitions_used, give the metric name and metric_version for every metric in your figures.
6. In sources, list the tables the tools read.
7. Write amounts in the headline and narrative exactly as returned, for example $5,188.20.
8. Decline, with status "declined", no figures, and a decline_reason, when a question asks for a forecast, for costs or metrics that are not listed above, for months without data, or for anything other than this retailer's profit.
"""

if LIVE:
    llm = ChatOpenAI(model=MODEL, base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5, temperature=0)
    agent = create_agent(model=llm, tools=TOOLS, system_prompt=AGENT_SYSTEM_PROMPT, response_format=ToolStrategy(AnalystAnswer))
else:
    print(SKIPPED)

First, the main question. Watch the messages: the model asks for the reporting period, the tools reply, it asks for more, and it finishes by filling in an `AnalystAnswer`.

The LLD has a diagram of the calls we expect for this question, in section 7, "Request trace for the main question". Open `docs/LLD.md` and compare it with what the model actually did.

In [41]:
def ask_agent(question):
    """Run the agent on one question and keep the tool log from the run."""
    tool_log.clear()
    start = time.time()
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": question}]}, {"recursion_limit": 40})
        answer, error, messages = result.get("structured_response"), None, result["messages"]
        if answer is None:
            error = "The model finished without a structured answer"
    except Exception as exception:
        answer, error, messages = None, f"{type(exception).__name__}: {exception}", []
    return {"technique": "Agent", "answer": answer, "tool_log": list(tool_log), "error": error, "seconds": round(time.time() - start, 1), "messages": messages}


if LIVE:
    agent_first = ask_agent(MAIN_QUESTION)
    for message in agent_first["messages"][1:]:
        message.pretty_print()
else:
    print(SKIPPED)

================================== Ai Message ==================================
Tool Calls:
  get_reporting_period (call_face6f6921e74adcac2819a6)
 Call ID: call_face6f6921e74adcac2819a6
  Args:
================================= Tool Message =================================
Name: get_reporting_period

{"as_of_date": "2026-11-01", "current_month": "2026-10", "previous_month": "2026-09", "available_months": ["2026-09", "2026-10"], "sources": [{"table": "reporting_period", "keys": ["as_of_date"]}]}
================================== Ai Message ==================================
Tool Calls:
  query_profit (query_profit_e6a5e73wjhzn)
 Call ID: query_profit_e6a5e73wjhzn
  Args:
    month: 2026-09
    group_by: product
    metric: contribution_profit
================================= Tool Message =================================
Name: query_profit

{"month": "2026-09", "group_by": "product", "metric": "contribution_profit", "metric_version": "1.0.0", "rows": [{"entity_type": "product", "en

In [42]:
if LIVE:
    print("Tools called:", [call["name"] for call in agent_first["tool_log"]])
    show_answer(agent_first)
else:
    print(SKIPPED)

Tools called: ['get_reporting_period', 'query_profit', 'query_profit', 'get_quality_flags', 'query_profit', 'get_quality_flags', 'compare_profit', 'compare_profit', 'get_metric_definition']


### In October 2026, Trail water bottle led products with $5,188.20 in contribution profit, and Hydration led categories with $13,358.06. Apparel fell sharply (-$5,914.87 vs. September), while Hydration rose slightly (+$376.06). Confidence is low due to missing Camp stove cost data and duplicate Trail water bottle orders.
Last month (2026-10) the top three products by contribution profit were Trail water bottle ($5,188.20), Headlamp ($4,250.00), and Rain jacket ($3,841.00). The top three categories were Hydration ($13,358.06), Apparel ($9,746.38), and Camp ($9,366.26, incomplete because Camp stove cost is missing). Compared with September 2026, Rain jacket fell $2,234.30 (-36.8%), Sun hat fell $1,632.50 (-76.3%), and Apparel fell $5,914.87 (-37.8%). Gainers included Sleeping bag (+$860.55, +35.3%), Hydration pack (+$628.25, +33.6%), Headlamp (+$538.00, +14.5%), and Trail water bottle (+$85.00, +1.7%). Hydration category rose $376.06 (+2.9%). Camp category change is unavailable because Camp stove's October product cost is missing.

| Figure | Value | Month |
|---|---|---|
| Trail water bottle, contribution profit | 5,188.20 USD | 2026-10 |
| Headlamp, contribution profit | 4,250.00 USD | 2026-10 |
| Rain jacket, contribution profit | 3,841.00 USD | 2026-10 |
| Hydration category, contribution profit | 13,358.06 USD | 2026-10 |
| Apparel category, contribution profit | 9,746.38 USD | 2026-10 |
| Camp category, contribution profit | 9,366.26 USD | 2026-10 |
| Rain jacket, contribution profit change vs September | -2,234.30 USD | 2026-10 |
| Rain jacket, contribution profit pct change vs September | -36.80 percent | 2026-10 |
| Sun hat, contribution profit change vs September | -1,632.50 USD | 2026-10 |
| Sun hat, contribution profit pct change vs September | -76.30 percent | 2026-10 |
| Sleeping bag, contribution profit change vs September | 860.55 USD | 2026-10 |
| Sleeping bag, contribution profit pct change vs September | 35.30 percent | 2026-10 |
| Headlamp, contribution profit change vs September | 538.00 USD | 2026-10 |
| Headlamp, contribution profit pct change vs September | 14.50 percent | 2026-10 |
| Trail water bottle, contribution profit change vs September | 85.00 USD | 2026-10 |
| Trail water bottle, contribution profit pct change vs September | 1.70 percent | 2026-10 |
| Apparel category, contribution profit change vs September | -5,914.87 USD | 2026-10 |
| Apparel category, contribution profit pct change vs September | -37.80 percent | 2026-10 |
| Hydration category, contribution profit change vs September | 376.06 USD | 2026-10 |
| Hydration category, contribution profit pct change vs September | 2.90 percent | 2026-10 |
| Camp category, contribution profit change vs September | unavailable USD | 2026-10 |
| Camp category, contribution profit pct change vs September | unavailable percent | 2026-10 |

- **Caveat QF-001:** No product cost is recorded for Camp stove in 2026-10, so its gross profit and contribution profit cannot be calculated. The Camp category total leaves it out.
- **Caveat QF-002:** Three orders for Trail water bottle appear twice in 2026-10. Its sales and profit may be overstated.
- **Caveat QF-003:** A 2026-09 refund for Sun hat is larger than the order it refunds. Its September net sales may be understated, which affects its change from September to October.

**Confidence: low.** QF-001 (high): No product cost is recorded for Camp stove in 2026-10, so its gross profit and contribution profit cannot be calculated. The Camp category total leaves it out. QF-002 (medium): Three orders for Trail water bottle appear twice in 2026-10. Its sales and profit may be overstated.

Now all the cases, scored exactly as in Part 4. Each question takes several model calls: one for each round of tool use and one to finish.

In [43]:
if LIVE:
    agent_results, agent_scores = run(ask_agent, cases)
    display(pd.DataFrame(agent_scores)[["case", *CHECKS, "wrong_figures"]])
else:
    print(SKIPPED)

Agent: 2 of 3 right in 58 seconds


,case,status,expected_figures,all_figures_true,flags,confidence,right,wrong_figures
0,E1,False,False,False,False,False,False,[]
1,E2,True,True,True,True,True,True,[]
2,E6,True,True,True,True,True,True,[]


## Part 6: Verify, compare, and move to production

### Putting it all together

Here are both approaches side by side. The numbers are the count of cases that passed each check.

In [44]:
def summarise(scores):
    """Pass counts per check, and the average time per question."""
    row = {"technique": scores[0]["technique"]}
    for name in CHECKS:
        row[name] = f"{sum(score[name] for score in scores)} / {len(scores)}"
    row["speed"] = f"{sum(score['seconds'] for score in scores) / len(scores):.1f} seconds a question"
    return row


if LIVE:
    display(pd.DataFrame([summarise(llm_scores), summarise(agent_scores)]))
else:
    print(SKIPPED)

,technique,status,expected_figures,all_figures_true,flags,confidence,right,speed
0,LLM,2 / 3,3 / 3,2 / 3,3 / 3,3 / 3,2 / 3,53.9 seconds a question
1,Agent,2 / 3,2 / 3,2 / 3,2 / 3,2 / 3,2 / 3,19.3 seconds a question


### The verifier: trust, but check

So far we scored answers against cases we prepared. A real leader asks questions we haven't prepared for. We need a check that works on any answer, with no answer key.

The agent gives us one for free: the tool log. If every figure in the answer equals a value that a tool returned in the same run, the figure wasn't invented. That is the first of the verifier checks in the LLD.

In [45]:
show_doc("LLD.md", "8. Verifier checks")

> **From `docs/LLD.md`: 8. Verifier checks**

| Check | Passes when |
|---|---|
| `figures_match_tools` | Every figure's value equals a value in the named tool's results for the same metric, month, and entity |
| `no_unsupported_numbers` | Every number in the headline and narrative matches a number in the run's tool results, allowing for rounding and currency formatting. Dates, identifiers, version numbers, and counts up to 12 are ignored |
| `sources_cited` | An answered response lists at least one source, and each source appears in the tool log |
| `definitions_stated` | Each metric used in a figure has a definition reference with the current version |
| `confidence_matches_rule` | An answer with figures has a confidence level equal to the level returned by `get_quality_flags` in that run (the worst one if it was called more than once) |
| `flags_mentioned` | Every flag that `get_quality_flags` returned in the run appears in the caveats by `flag_id`. If the agent fetched a flag, it must report it |
| `decline_is_clean` | A declined answer has no figures and has a reason |

Here are the two most important checks, written simply. `figures_match_tools` looks up each figure in the run's own tool log. `confidence_matches_rule` compares the answer's confidence with the level the quality tool returned.

In [46]:
def facts_from_log(log):
    """Every figure the tools returned in one run, keyed like the facts in Part 4."""
    found = {}
    for call in log:
        result = call["result"]
        if result is None:
            continue
        if call["name"] == "query_profit":
            for row in result["rows"]:
                for term in ALL_TERMS:
                    found.setdefault((term, "value", row["month"], row["entity_id"]), []).append(row[term])
        elif call["name"] == "compare_profit":
            for row in result["rows"]:
                found.setdefault((result["metric"], "value", result["month"], row["entity_id"]), []).append(row["current"])
                found.setdefault((result["metric"], "value", result["previous_month"], row["entity_id"]), []).append(row["previous"])
                found.setdefault((result["metric"], "change", result["month"], row["entity_id"]), []).append(row["change"])
                found.setdefault((result["metric"], "pct_change", result["month"], row["entity_id"]), []).append(row["pct_change"])
    return found


def verify(result):
    """Check an answer against the tool log of the run that produced it."""
    answer, log = result["answer"], result["tool_log"]
    returned = facts_from_log(log)
    unsupported = [figure.label for figure in answer.figures if not any(same_value(figure.value, value) for value in returned.get((figure.metric, figure.kind, figure.month, figure.entity_id), []))]
    levels = [call["result"]["confidence_level"] for call in log if call["name"] == "get_quality_flags" and call["result"]]
    rule = max(levels, key=CONFIDENCE_ORDER.index) if levels else None
    stated = answer.confidence.level if answer.confidence else None
    return {
        "figures_match_tools": not unsupported,
        "figures not returned by any tool": unsupported,
        "confidence_matches_rule": not answer.figures or stated == rule,
        "confidence": f"answer says {stated}, the rule gives {rule}",
    }


if LIVE and agent_first["answer"]:
    display(verify(agent_first))
else:
    print(SKIPPED)

{'figures_match_tools': True,
 'figures not returned by any tool': [],
 'confidence_matches_rule': True,
 'confidence': 'answer says low, the rule gives low'}

### From notebook to production

Everything above is brainstorming code: written top to bottom, in one file, to work out the solution. The same logic has been rebuilt as production code in this milestone's `retail_profit_stack/` package, with tests in `tests/`.

| In this notebook | In production | What changed |
|---|---|---|
| pandas DataFrames | `store.py`, DuckDB | The tables are loaded once and checked against their contract |
| Tool functions returning dicts | `tools.py`, `schemas.py` | Results are typed Pydantic models |
| `ask_agent` | `agent.py` | One function, `run_analyst`, returns the answer, the tool log, and the verification |
| `verify`, 2 checks | `verify.py`, 7 checks | Adds checks on numbers in the text, sources, definitions, flags, and declined answers |
| `score` and `run` | `evaluation.py` | The same scoring, plus every verifier check |

The production tools must return what the notebook's tools return. Let's check a few.

In [47]:
sys.path.insert(0, str(MILESTONE))
from retail_profit_stack import tools as production_tools

notebook_rows = query_profit(LAST_MONTH, "category")["rows"]
production_rows = production_tools.query_profit(LAST_MONTH, "category").model_dump()["rows"]
print("Category totals match:", notebook_rows == production_rows)

notebook_change = compare_profit(LAST_MONTH, PREVIOUS_MONTH, "product")["rows"]
production_change = production_tools.compare_profit(LAST_MONTH, PREVIOUS_MONTH, "product").model_dump()["rows"]
print("Product changes match:", notebook_change == production_change)

Category totals match: True
Product changes match: True


And here is the production verifier, with all seven checks, run on the answer our notebook agent gave. No model is called here: the verifier is plain code.

In [48]:
from retail_profit_stack.schemas import AnalystAnswer as ProductionAnswer
from retail_profit_stack.schemas import ToolCall
from retail_profit_stack.verify import verify_answer

if LIVE and agent_first["answer"]:
    report = verify_answer(
        ProductionAnswer(**agent_first["answer"].model_dump()),
        [ToolCall(**call) for call in agent_first["tool_log"]],
    )
    print("Verified:", report.passed)
    display(pd.DataFrame([check.model_dump() for check in report.checks]))
else:
    print(SKIPPED)

Verified: False


,name,passed,detail
0,figures_match_tools,True,22 figures match tool results
1,no_unsupported_numbers,False,"Numbers in the text that no tool returned: [2026.0, 2026.0]"
2,sources_cited,True,"Cites ['metric_definitions', 'monthly_product_profit', 'quality_flags']"
3,definitions_stated,True,Every metric used has its definition and version stated
4,confidence_matches_rule,True,Answer says low; the rule gives low
5,flags_mentioned,True,All 3 returned flags are in the caveats
6,decline_is_clean,True,Not a declined answer


To run the production tests, open a terminal in this milestone's folder and run:

```
uv run pytest
```

The tests use a scripted fake model, so they need no key and give the same result every time.

> ### Your assignment
>
> This lab is about using AI to meet a business requirement, and knowing how far to trust it.
>
> `openrouter/free` picks a different free model for each request. The free models on OpenRouter each have an id that ends in `:free`.
>
> 1. Set `MODEL_NAME` in your `.env` to a specific free model. Which one gets the most cases right with the agent? Which one fails `all_figures_true` most often on the single call?
> 2. Add a case of your own to `evals/m1_cases.json`: a question, and the figures a correct answer must contain. Work out the expected values with pandas first, as we did in Part 2.
> 3. Read the agent's system prompt again. Remove rule 3 (the one about quality flags) and re-run Part 5. Which checks start to fail? Then put it back.

> ### What you built
>
> You took a business question from a requirements document to a checked answer. You saw a model draft metric definitions and raise ambiguities, tried the question as a single call with the data in the prompt, then as an agent with tools, and measured both.
>
> The lesson to carry forward: let the model do what it's good at (understanding questions, choosing what to look up, explaining), keep every number in code, and check the result with code.
>
> Everything behind the tools is still mocked. In Milestone 2 we replace the precomputed profit table with real transformations, behind the same six tools, and this same evaluation set has to keep passing.